# Hidden Shift with native Boolean programs

Recover a hidden bit shift in three original-size scenarios. User code expresses Boolean functions; synthesis chooses reversible gates and clean workspace. All results below use local execution and independent Circuit validation. Optional dependencies: SymPy and Matplotlib.


## Phase functions, interference and reconstruction

The visible four-bit program composes the shifted and unshifted bent-function
phases with Hadamard transforms. These are quantum phase expressions; the
compiler constructs their reversible realization. Interference converts the
hidden shift into a named output, rather than reading a phase from a histogram.

The later dual and no-dual routes retain the original sixteen-bit problem and
2048 shots. Their reusable qfuncs live in [workflow.py](workflow.py); the no-dual
route also returns an indicator used in classical GF(2) reconstruction. They
are not replaced by the smaller visible example.

Try removing the shift in the four-bit phase expression and predict the returned
shift. Changing the sixteen-bit permutation requires updating its inverse and
independent reconstruction; those are algorithm semantics, not routing choices.


In [ ]:
import math
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import (
    Output,
    QArray,
    QBit,
    allocate,
    hadamard_transform,
    phase,
    qfunc,
)


@qfunc
def simple_shift(s: Output[QArray[QBit]]) -> None:
    allocate(4, s)
    hadamard_transform(s)
    phase((1 - s[0]) * s[1] + s[2] * s[3], math.pi)
    hadamard_transform(s)
    phase(s[0] * s[1] + s[2] * s[3], math.pi)
    hadamard_transform(s)


visible_program = synth.synthesize(
    simple_shift, max_qubits=4, max_workspace=0, memory_limit_bytes=1024**2
)
print(
    "Four-bit named state:",
    dict(visible_program.probabilities(fq.run(visible_program), registers=("s",))),
)
from io import BytesIO

from IPython.display import Image, display
from matplotlib.figure import Figure

from examples.synthesis.algorithms.number_theory_and_cryptography.hidden_shift.overview.workflow import (
    PERMUTATION,
    SHIFT,
    recover_shift,
    run_estimate,
)

simple_result = run_estimate("simple", seed=1500)
print("Four-bit shift:", simple_result.recovered)
print("2048 observations:", simple_result.counts)
print(simple_result.resources)


## The original sixteen-bit bent function

The seed-1 permutation defines the paired terms. The dual adds the product of all eight first-half bits. The supplied shifts are exactly 1, 3 and 9.


In [ ]:
print("Permutation:", PERMUTATION)
print("Shifted positions:", SHIFT)
dual_result = run_estimate("dual", seed=1501)
print("Recovered shift:", dual_result.recovered)
print("2048 observations:", dual_result.counts)
print(dual_result.resources)


## Recover the shift without its dual

The indicator records a binary linear equation for each observed target. Its Z phase must retain the absolute phase convention. Synthesis allocates four clean work bits in addition to the sixteen target bits and indicator.


In [ ]:
no_dual_result = run_estimate("no_dual", seed=1502)
equations = [
    (tuple((word // 2) >> bit & 1 for bit in range(16)), word & 1)
    for word, count in no_dual_result.counts
]
recovered = recover_shift(equations)
expected = tuple(int(bit in SHIFT) for bit in range(16))
assert recovered == expected == dual_result.recovered
print("GF(2) recovered shift:", recovered)
print("Distinct sampled equations:", len(equations))
print(no_dual_result.resources)


## Check state, workspace and actual samples

Each run compares every absolute complex amplitude and the same-seed samples against an independent Circuit. Binary rank and inversion use SymPy over GF(2), not floating-point linear algebra. The source used unseeded hosted sampling; these local seeds give reproducibility rather than an identical-hosted-output claim.


In [ ]:
for estimate in (simple_result, dual_result, no_dual_result):
    print(
        estimate.route,
        "shots",
        estimate.shots,
        "state error",
        estimate.maximum_state_error,
        "dirty work",
        estimate.dirty_workspace_probability,
    )
    assert sum(count for word, count in estimate.counts) == 2048


In [ ]:
indicator_counts = [
    sum(count for word, count in no_dual_result.counts if word & 1 == indicator)
    for indicator in (0, 1)
]
figure = Figure(figsize=(6, 3), layout="constrained")
axis = figure.subplots()
axis.bar(["indicator 0", "indicator 1"], indicator_counts)
axis.set_ylabel("Actual observations")
axis.set_title("No-dual Hidden Shift: 2048 shots")
with BytesIO() as buffer:
    figure.savefig(buffer, format="png", dpi=140)
    display(Image(data=buffer.getvalue()))
